In [105]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

In [106]:
# Connecting to Workspace desktop session to obtain data
import lseg.data as ld

ld.open_session()

<lseg.data.session.Definition object at 0x2aa894809e0 {name='workspace'}>

In [ ]:
# To evaluate MultiIndex data later in code
idx = pd.IndexSlice

In [ ]:
# Loading in current Russell 1000 index constituents CSV file - pulled from Workspace
russ = pd.read_csv('russ1k_constituents.csv')
russ.dropna(inplace=True)

rics = russ['RIC'].to_list()

russ_IPOs = ld.get_data(
    universe=rics,
    fields=['TR.IPODate']
)

# Using 250 instruments with the oldest IPO dates
russ_250_oldest = russ_IPOs.sort_values(by='IPO Date',ascending=True).head(250)['Instrument'].to_list()

In [108]:
# Configuration class to define parameters used throughout the code
class Config:
    START_DATE = '2010-01-04'
    END_DATE = '2025-12-31'
    RIC_LIST = russ_250_oldest

In [ ]:
# Constructing full DataFrame for backtest

# Starting out with initial temporary DataFrames to concatenate with other DataFrames as data gets pulled for each instrument
first_RIC = Config.RIC_LIST[0]

temp_price_mcap = ld.get_history(
    universe = [first_RIC],
    fields = [
        'TR.ClosePrice',
        'TR.CompanyMarketCap(ShType=FFL)'
        ],
    start = Config.START_DATE,
    end = Config.END_DATE
)

temp_div_bb = ld.get_history(
    universe = [first_RIC],
    fields = [
        'TR.F.ComStockBuybackNet(ReportType=Prelim, ReportingState=Orig)',
        'TR.F.DivPaidCashTotCF(ReportType=Prelim, ReportingState=Orig)'
    ],
    start = Config.START_DATE,
    end = Config.END_DATE
)

# Full DataFrame that will be used as starting point
df = temp_price_mcap.join(temp_div_bb, how='outer').ffill()
df = pd.concat([df], axis=1, keys=[first_RIC]) 


# Looping through list of RICs and obtaining data for each
count = 0
for RIC in Config.RIC_LIST:
    
    count += 1
    print(f'Loop {count}: {RIC}')
    
    price_mcap = ld.get_history(
        universe = [RIC],
        fields = [
            'TR.ClosePrice',
            'TR.CompanyMarketCap(ShType=FFL)'
            ],
        start = Config.START_DATE,
        end = Config.END_DATE
    )

    # Ensuring there is sufficient price and market cap data for each instrument
    if price_mcap.loc['2010-01-04':].isna().sum().sum() > 25:
        print(f'Missing data for {RIC}')

    div_bb = ld.get_history(
    universe = [RIC],
    fields = [
        'TR.F.ComStockBuybackNet(ReportType=Prelim, ReportingState=Orig)',
        'TR.F.DivPaidCashTotCF(ReportType=Prelim, ReportingState=Orig)'
    ],
    start = Config.START_DATE,
    end = Config.END_DATE
    )
    
    temp_df = price_mcap.join(div_bb, how='outer').ffill()
    temp_df = pd.concat([temp_df], axis=1, keys=[RIC]) 

    df = pd.concat([df, temp_df], axis=1)

# Stacking into MultiIndex format
df = df.loc['2010-01-04':]
df = df.stack(level=0)
df.index.names = ['Date', 'RIC']


df.head()
    

In [227]:
# Adding in sector data for each instrument
sector_df = ld.get_data(
    universe = Config.RIC_LIST, 
    fields = ['TR.TRBCEconomicSector']
    )

sector_df = sector_df.rename(
    columns = {'Instrument': 'RIC', 
               'TRBC Economic Sector Name': 'sector'}
    )

sector_df = sector_df.set_index('RIC')

In [228]:
# Concatenating data and cleaning up column names
new_cols = ['close', 'market_cap', 'buybacks_ttm', 'dividends_ttm']
col_dict = {}
for old_col, new_col in zip(df.columns, new_cols):
    col_dict[old_col] = new_col

df = df.rename(columns=col_dict)
df = df.join(sector_df, how='left', on='RIC')

In [ ]:
# Cleaning up number formatting and combining dividends and buybacks into one column
df['buybacks_ttm'].fillna(0, inplace=True)
df[['close', 'market_cap']] = df[['close', 'market_cap']].astype(float)
df['buybacks_ttm'] = pd.to_numeric(df['buybacks_ttm'], errors='coerce').fillna(0).astype(float)
df['dividends_ttm'] = pd.to_numeric(df['dividends_ttm'], errors='coerce').fillna(0).astype(float)
df['total_payout_ttm'] = df['buybacks_ttm'] + df['dividends_ttm']


In [ ]:
# Dropping constituents with missing data
missing_data = ['WDC.OQ', 'AON.N', 'CRH.N', 'BFa.N']
df = df.drop(missing_data, level=1)

df.info()

In [ ]:
# Checking how many rows of data is available on each date
date_dict = {}
date_set = set(df.index.get_level_values(0))

for date in date_set:
    date_dict[str(date)] = len(df.loc[idx[date, :], :])

date_count_df = pd.Series(date_dict)

date_count_df.value_counts()

In [ ]:
# Looking into why some dates don't have data for all the stocks
bad_dates_orig = date_count_df[date_count_df != 246].index
bad_dates = pd.to_datetime(bad_dates_orig)

bad_dates

In [ ]:
# Checking if the dates are weekends
weekday_dates, weekend_dates = [], []

for temp_date in bad_dates_orig:
    date = pd.to_datetime(temp_date)
    day = pd.Timestamp(date).strftime('%A')
    if day not in ('Saturday', 'Sunday'):
        weekday_dates.append((f'{day}: {date}', date_count_df[temp_date]))

    else:
        weekend_dates.append(f'{day}: {date}')

In [ ]:
# Confirmed most dates with missing data are weekends and others are holidays - removing from DataFrame
df = df[~df.index.get_level_values('Date').isin(pd.to_datetime(bad_dates))]

In [ ]:
# Quick check of final data set
df.index.names = ['date', 'RIC']
df.info()

In [322]:
# Saving DataFrame to a pickle file
df.to_pickle('russ_250_df.pkl')